In [76]:
import os
import sys
import numpy as np
import pandas as pd
import networkx as nx
import plotly.express as px
from pathlib import Path

# Make sure the GlobalModel utils and graph_plot are importable
NOTEBOOK_DIR = Path(os.getcwd())
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from utils import compute_distances_allvsall
from graph_plot import plot_networkx_plotly


In [77]:
# ── Config ─────────────────────────────────────────────────────────────────
DATA_PATH  = str(NOTEBOOK_DIR / '../../dataset/data_smooth_erratic.feather')
DATE_COL   = 'date'
TARGET_COL = 'value'

# Distance metric to use — pick one:
# 'euclidean' | 'manhattan' | 'cid' | 'dtw' | 'hamming' | 'amplitude_offset'
# 'slope_consistency' | 'lorentzian' | 'sbd' | 'msm' | 'edr' | 'lcss'
# 'twed' | 'erp' | 'stid' | 'phase_invariance'
METRIC = 'amplitude_offset'

# DISTANCE_THRESHOLD is set AFTER inspecting the distribution in the next cells.
# Lower = stricter (fewer edges). Set to None here; assign after seeing the histogram.




In [78]:
val_size = 30
forecast_horizon = 153
train_size = 761 - val_size - forecast_horizon  # 455
lookback_window = 30

In [79]:
# ── Load dataset & build wide pivot (item_id × date) ──────────────────────
df = pd.read_feather(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values([DATE_COL, 'item_id']).reset_index(drop=True)

df_wide = (
    df.pivot_table(index='item_id', columns=DATE_COL, values=TARGET_COL, aggfunc='sum')
    .fillna(0)
)

# Restrict to the training window only — distances must not see val/test data
df_wide = df_wide.iloc[:, :train_size]

# Optional category labels for colouring nodes in the plot
cat_labels_dict = (
    df.drop_duplicates('item_id').set_index('item_id')['cat_label'].to_dict()
    if 'cat_label' in df.columns else {}
)

item_ids = df_wide.index.tolist()
print(f"Loaded: {len(item_ids)} products × {df_wide.shape[1]} time steps (train only)")


Loaded: 972 products × 578 time steps (train only)


In [80]:
# ── Compute all-vs-all distance matrix (GPU-accelerated if available) ──────
all_ts = df_wide.values.astype(np.float32)   # (N, T)

print(f"Computing {METRIC} distances for {len(item_ids)} × {len(item_ids)} pairs...")
dist_matrix = compute_distances_allvsall(all_ts, metric=METRIC)

print(f"Done.  Matrix shape: {dist_matrix.shape}")
print(f"Value range (excl. diagonal): [{np.triu(dist_matrix, k=1)[np.triu(dist_matrix, k=1) > 0].min():.4f}, "
      f"{dist_matrix.max():.4f}]")


Computing amplitude_offset distances for 972 × 972 pairs...
Done.  Matrix shape: (972, 972)
Value range (excl. diagonal): [13.7708, 39.5991]


In [81]:
# ── Distribution of pairwise distances ────────────────────────────────────
# Extract upper-triangle values only (each pair counted once, no self-loops)
upper_idx = np.triu_indices(len(item_ids), k=1)
pairwise_dists = dist_matrix[upper_idx]

# Summary statistics
p1, p2, p3, p5, p10 = np.percentile(pairwise_dists, [1,2,3,5,10])
print(f"Pairwise {METRIC} distances  (N={len(pairwise_dists):,} pairs)")
print(f"  min={pairwise_dists.min():.4f}  p1={p1:.4f}  p2={p2:.4f}  p3={p3:.4f}  "
      f"p5={p5:.4f}  p10={p10:.4f}  max={pairwise_dists.max():.4f}")
'''
# Interactive histogram
fig = px.histogram(
    x=pairwise_dists,
    nbins=100,
    title=f'Distribution of pairwise {METRIC} distances ({len(item_ids)} products)',
    labels={'x': f'{METRIC} distance', 'y': 'Count'},
    opacity=0.8,
)
# Overlay percentile reference lines
for pct_val, label, colour in [
    (p1, 'p1', 'green'),
    (p2, 'p2', 'blue'),
    (p3, 'p3', 'purple'),
    (p5, 'p5', 'orange'),
    (p10, 'p10', 'red'),
]:
    fig.add_vline(x=pct_val, line_dash='dash', line_color=colour,
                  annotation_text=f'{label}={pct_val:.3f}', annotation_position='top right')

fig.update_layout(bargap=0.02, xaxis_title=f'{METRIC} distance', yaxis_title='Pair count')
fig.show()
print("\nInspect the histogram above, then set DISTANCE_THRESHOLD in the next cell.")
'''

Pairwise amplitude_offset distances  (N=471,906 pairs)
  min=13.7708  p1=28.5151  p2=29.3534  p3=29.8070  p5=30.3269  p10=30.9825  max=39.5991


'\n# Interactive histogram\nfig = px.histogram(\n    x=pairwise_dists,\n    nbins=100,\n    title=f\'Distribution of pairwise {METRIC} distances ({len(item_ids)} products)\',\n    labels={\'x\': f\'{METRIC} distance\', \'y\': \'Count\'},\n    opacity=0.8,\n)\n# Overlay percentile reference lines\nfor pct_val, label, colour in [\n    (p1, \'p1\', \'green\'),\n    (p2, \'p2\', \'blue\'),\n    (p3, \'p3\', \'purple\'),\n    (p5, \'p5\', \'orange\'),\n    (p10, \'p10\', \'red\'),\n]:\n    fig.add_vline(x=pct_val, line_dash=\'dash\', line_color=colour,\n                  annotation_text=f\'{label}={pct_val:.3f}\', annotation_position=\'top right\')\n\nfig.update_layout(bargap=0.02, xaxis_title=f\'{METRIC} distance\', yaxis_title=\'Pair count\')\nfig.show()\nprint("\nInspect the histogram above, then set DISTANCE_THRESHOLD in the next cell.")\n'

In [82]:
# ── Set threshold here after inspecting the histogram ─────────────────────
# Edges are kept where distance <= DISTANCE_THRESHOLD (smaller = more similar).
# Good starting points are p25 or p50 depending on how many connections you want.
DISTANCE_THRESHOLD = 20   # ← change to any value, e.g. p50, p75, or a fixed float
print(f"DISTANCE_THRESHOLD set to: {DISTANCE_THRESHOLD:.4f}")


DISTANCE_THRESHOLD set to: 20.0000


In [83]:
# ── Apply threshold → keep only connected products ─────────────────────────
N = len(item_ids)

# Vectorised: keep upper-triangle pairs where distance <= threshold
mask = np.triu(dist_matrix <= DISTANCE_THRESHOLD, k=1)  # exclude self-loops
rows, cols = np.where(mask)

G_full = nx.Graph()
G_full.add_nodes_from(item_ids)
for i, j in zip(rows, cols):
    G_full.add_edge(item_ids[i], item_ids[j], weight=float(dist_matrix[i, j]))

# Sub-graph: only nodes with at least one edge
connected_nodes = [n for n, d in G_full.degree() if d > 0]
G = G_full.subgraph(connected_nodes).copy()

# Attach category label to each node (used by plot_networkx_plotly for colouring)
for node in G.nodes():
    G.nodes[node]['cat_label'] = cat_labels_dict.get(node, 'Unknown')

isolated = N - G.number_of_nodes()
print(f"Distance threshold = {DISTANCE_THRESHOLD:.4f}  |  metric = {METRIC}")
print(f"  Connected products : {G.number_of_nodes():>5d}  ({isolated} isolated, removed)")
print(f"  Edges              : {G.number_of_edges():>5d}")


Distance threshold = 20.0000  |  metric = amplitude_offset
  Connected products :    34  (938 isolated, removed)
  Edges              :    40


In [84]:
# ── Interactive plot (only connected products are shown) ───────────────────
plot_networkx_plotly(
    G,
    title=(
        f'All-vs-All Product Distance  ({METRIC} ≤ {DISTANCE_THRESHOLD:.4f})'
        f'  —  {G.number_of_nodes()} connected products  |  {G.number_of_edges()} edges'
    ),
)
